# Predicting high-NO₂ days in Madrid

## 1. Problem description
Nitrogen dioxide (NO₂) comes mainly from road traffic and is the pollutant that most often puts Madrid above air-quality limits. When a high-pollution day is coming, the city can activate its pollution protocol (speed limits, parking restrictions, public warnings), but to work well those measures need to be announced **the day before**.

**Goal:** using what is known today, predict whether **tomorrow's daily mean NO₂** at a traffic station will be above **40 µg/m³**.

- **Task type:** binary classification (1 = high-NO₂ day, 0 = normal day).
- **Why 40 µg/m³:** it is the EU annual limit value for NO₂. A day above it is a day that pushes the station towards breaking the yearly limit. The threshold is a project choice and can be changed in the configuration cell.
- **Imbalance:** high days are a minority, so accuracy alone would be misleading.

## 2. What makes this harder than a random tabular dataset
1. **Time matters.** The data is a time series, so a random train/test split would let the model "see the future". We split by year and validate with `TimeSeriesSplit`.
2. **Only use what you would know in time.** Every feature is built from today and earlier, except tomorrow's weather, which stands in for the weather forecast (see Limitations).
3. **A real baseline.** Before any ML, we check the naive rule *"tomorrow will be like today"* (persistence). A model is only useful if it beats it.

## 3. Data sources
| Data | Source | How to get it |
|---|---|---|
| Daily air quality per station (NO₂, NO, PM10, O₃…) | Ayuntamiento de Madrid open data portal, dataset *"Calidad del aire. Datos diarios desde 2001"* | Download one **CSV per year** (e.g. 2015 → last complete year) and upload them in the cell below |
| Daily weather (temperature, rain, wind) | Open-Meteo historical weather API (free, no key) | Downloaded automatically |
| Public holidays in Madrid | `holidays` Python package | Automatic |

Each air-quality file has one row per **station × pollutant × month**, with the daily values in columns `D01…D31` and a validity flag in `V01…V31` (`V` = valid). The loader below handles this format.

In [ ]:
!pip install -q xgboost holidays

In [ ]:
import glob, json, math, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests, holidays

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# ---------------- Configuration ----------------
DATA_DIR   = "data"          # folder with the yearly CSV files
STATION    = 8               # 8 = Escuelas Aguirre (traffic station). Change to try others.
THRESHOLD  = 40.0            # µg/m³, daily mean NO2 considered "high"
LAT, LON   = 40.4217, -3.6823  # approx. location of Escuelas Aguirre (for weather)
MAG = {"NO": 7, "NO2": 8, "PM10": 10, "O3": 14}   # magnitude codes used in the files
RANDOM_STATE = 42
os.makedirs(DATA_DIR, exist_ok=True)

### Upload the yearly CSV files
In Colab, run the cell and select all the yearly files at once. Outside Colab, just copy them into the `data/` folder.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not glob.glob(f"{DATA_DIR}/*.csv"):
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(DATA_DIR, name), "wb") as f:
            f.write(content)

paths = sorted(glob.glob(f"{DATA_DIR}/*.csv"))
print(f"{len(paths)} files found:", [os.path.basename(p) for p in paths])
assert paths, "Upload at least one yearly CSV file from the Madrid open data portal."

## 4. Loading and reshaping the data
The files are in a *wide* format (one column per day of the month). We turn them into a *long* table with one row per station, pollutant and date, keeping only values flagged as valid.

In [ ]:
def read_any(path):
    # The portal has used different separators and encodings over the years
    for enc in ("utf-8", "latin-1"):
        try:
            df = pd.read_csv(path, sep=None, engine="python", dtype=str, encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    df.columns = [c.strip().upper().replace("AÑO", "ANO") for c in df.columns]
    return df

def load_daily(paths):
    raw = pd.concat([read_any(p) for p in paths], ignore_index=True)
    day_cols = [f"D{d:02d}" for d in range(1, 32)]
    val_cols = [f"V{d:02d}" for d in range(1, 32)]
    missing = [c for c in ["ESTACION", "MAGNITUD", "ANO", "MES"] + day_cols if c not in raw.columns]
    assert not missing, f"Unexpected file format, missing columns: {missing}"

    keys = ["ESTACION", "MAGNITUD", "ANO", "MES"]
    vals = raw.melt(id_vars=keys, value_vars=day_cols, var_name="DIA", value_name="VALUE")
    flags = raw.melt(id_vars=keys, value_vars=val_cols, var_name="DIA", value_name="FLAG")
    vals["DIA"] = vals["DIA"].str[1:].astype(int)
    flags["DIA"] = flags["DIA"].str[1:].astype(int)
    long = vals.merge(flags, on=keys + ["DIA"])

    long = long[long["FLAG"].str.strip().str.upper() == "V"].copy()
    long["VALUE"] = pd.to_numeric(long["VALUE"].str.replace(",", "."), errors="coerce")
    long["date"] = pd.to_datetime(dict(year=long["ANO"].astype(int), month=long["MES"].astype(int),
                                       day=long["DIA"]), errors="coerce")
    long = long.dropna(subset=["date", "VALUE"])
    long["station"] = long["ESTACION"].astype(int)
    long["magnitude"] = long["MAGNITUD"].astype(int)
    return long[["date", "station", "magnitude", "VALUE"]]

long = load_daily(paths)
print(long.shape)
print("Stations:", sorted(long["station"].unique()))
print("Years:", long["date"].dt.year.min(), "→", long["date"].dt.year.max())

In [ ]:
# One row per day for the chosen station, one column per pollutant
inv = {v: k for k, v in MAG.items()}
st = (long[(long["station"] == STATION) & (long["magnitude"].isin(MAG.values()))]
      .pivot_table(index="date", columns="magnitude", values="VALUE"))
st = st.rename(columns=inv)
st = st.asfreq("D")   # regular calendar: missing days become NaN instead of disappearing
print(st.shape)
print("Missing values per pollutant (%):")
print((st.isna().mean() * 100).round(1))
st.head()

## 5. Weather and calendar data
NO₂ builds up when the air is still: cold, windless, dry days with temperature inversions. Rain and wind clean the air. That is why weather is likely to be key.

In [ ]:
def get_weather(start, end, lat=LAT, lon=LON):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = dict(latitude=lat, longitude=lon, start_date=start, end_date=end, timezone="Europe/Madrid",
                  daily="temperature_2m_max,temperature_2m_min,precipitation_sum,"
                        "wind_speed_10m_max,wind_direction_10m_dominant,shortwave_radiation_sum")
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    w = pd.DataFrame(r.json()["daily"])
    w["time"] = pd.to_datetime(w["time"])
    return w.set_index("time").rename(columns={
        "temperature_2m_max": "tmax", "temperature_2m_min": "tmin", "precipitation_sum": "rain",
        "wind_speed_10m_max": "wind", "wind_direction_10m_dominant": "wind_dir",
        "shortwave_radiation_sum": "radiation"})

weather = get_weather(st.index.min().strftime("%Y-%m-%d"), st.index.max().strftime("%Y-%m-%d"))
df = st.join(weather, how="left")
years = range(df.index.year.min(), df.index.year.max() + 2)
md_holidays = holidays.Spain(subdiv="MD", years=list(years))
df["holiday"] = df.index.isin(pd.to_datetime(list(md_holidays.keys()))).astype(int)
df.head()

## 6. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df["NO2"].plot(ax=ax, lw=0.6, alpha=0.8)
df["NO2"].rolling(30).mean().plot(ax=ax, lw=2, label="30-day mean")
ax.axhline(THRESHOLD, color="red", ls="--", label=f"Threshold {THRESHOLD:.0f} µg/m³")
ax.set_title(f"Daily mean NO₂ at station {STATION}"); ax.set_ylabel("µg/m³"); ax.legend()
plt.show()

**What to look for:** a strong yearly cycle (higher in winter), a long-term downward trend, and the 2020 lockdown drop.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=df.index.month, y=df["NO2"], ax=axes[0], color="#8fb3d9")
axes[0].set_title("NO₂ by month"); axes[0].set_xlabel("Month")
names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
sns.boxplot(x=df.index.dayofweek, y=df["NO2"], ax=axes[1], color="#8fb3d9")
axes[1].set_xticklabels(names); axes[1].set_title("NO₂ by day of week"); axes[1].set_xlabel("")
for a in axes: a.axhline(THRESHOLD, color="red", ls="--")
plt.tight_layout(); plt.show()

In [ ]:
high = (df["NO2"] > THRESHOLD)
share = high.groupby(df.index.year).mean().mul(100).round(1)
share.plot(kind="bar", figsize=(10, 3.5), color="#c0504d")
plt.title("Share of high-NO₂ days per year (%)"); plt.ylabel("%"); plt.xlabel("")
plt.show()
print(f"Overall share of high days: {high.mean():.1%}")

In [ ]:
cols = [c for c in ["NO2", "NO", "PM10", "O3", "tmax", "tmin", "rain", "wind", "radiation"] if c in df]
plt.figure(figsize=(8, 6))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlations"); plt.show()

**Expected findings to confirm with your data:** NO₂ correlates strongly with NO (same source, traffic), negatively with wind and rain, and negatively with O₃ (ozone reacts with NO in the city).

## 7. Feature engineering
Each row is **day *t***, and the target is whether **day *t+1*** is a high-NO₂ day.

| Group | Features | Known at time *t*? |
|---|---|---|
| Pollution today and recent days | NO₂ today, NO₂ 1-3 days ago, 7-day mean, NO, PM10, O₃ today | Yes |
| Weather tomorrow | temperature, rain, wind, wind direction (as sin/cos), radiation | Via the weather forecast |
| Weather today | rain and wind today | Yes |
| Calendar tomorrow | day of week, weekend, holiday, month (as sin/cos) | Yes |

Cyclic variables (month, wind direction) are encoded with sine and cosine so that December is close to January and 350° is close to 10°.

In [ ]:
def build_features(df):
    f = pd.DataFrame(index=df.index)
    no2 = df["NO2"]
    f["no2_today"] = no2
    for k in (1, 2, 3):
        f[f"no2_lag{k}"] = no2.shift(k)
    f["no2_mean7"] = no2.rolling(7, min_periods=4).mean()
    for c in ["NO", "PM10", "O3"]:
        if c in df: f[f"{c.lower()}_today"] = df[c]
    f["rain_today"] = df["rain"]
    f["wind_today"] = df["wind"]

    nxt = df.shift(-1)                          # tomorrow's values
    for c in ["tmax", "tmin", "rain", "wind", "radiation"]:
        f[f"{c}_tomorrow"] = nxt[c]
    rad = np.deg2rad(nxt["wind_dir"])
    f["wind_dir_sin_tomorrow"], f["wind_dir_cos_tomorrow"] = np.sin(rad), np.cos(rad)

    tomorrow = df.index + pd.Timedelta(days=1)
    f["dow_tomorrow"] = tomorrow.dayofweek
    f["weekend_tomorrow"] = (tomorrow.dayofweek >= 5).astype(int)
    f["holiday_tomorrow"] = nxt["holiday"]
    f["month_sin"] = np.sin(2 * np.pi * tomorrow.month / 12)
    f["month_cos"] = np.cos(2 * np.pi * tomorrow.month / 12)

    y = (no2.shift(-1) > THRESHOLD).astype(float)
    y[no2.shift(-1).isna()] = np.nan             # unknown target → drop later
    return f, y

X_all, y_all = build_features(df)
keep = y_all.notna() & X_all["no2_today"].notna()
X_all, y_all = X_all[keep], y_all[keep].astype(int)
print(X_all.shape, f"positives: {y_all.mean():.1%}")
X_all.tail()

## 8. Evaluation methodology
### Temporal split
- **Train:** all years except the last two.
- **Validation:** second-to-last year, used to tune the decision threshold.
- **Test:** last year, used only once at the end.

A random, stratified split (as in a typical classification project) would mix days from the same weeks into train and test, and the model would look better than it really is.

### Metrics
- **PR-AUC (average precision):** the main metric. With imbalanced classes it is more informative than ROC-AUC.
- **F1, precision and recall** at the chosen threshold.
- **ROC-AUC** for reference.

In [ ]:
last_year = X_all.index.year.max()
VAL_YEAR, TEST_YEAR = last_year - 1, last_year
tr = X_all.index.year < VAL_YEAR
va = X_all.index.year == VAL_YEAR
te = X_all.index.year == TEST_YEAR
X_train, y_train = X_all[tr], y_all[tr]
X_val, y_val = X_all[va], y_all[va]
X_test, y_test = X_all[te], y_all[te]
for n, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{n:5s}: {len(yy):5d} days, {yy.mean():.1%} high")
print(f"Train ≤ {VAL_YEAR-1}, validation = {VAL_YEAR}, test = {TEST_YEAR}")

## 9. Baseline: "tomorrow will be like today"

In [ ]:
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix)

def scores(y, prob, thr):
    pred = (prob >= thr).astype(int)
    return dict(PR_AUC=average_precision_score(y, prob), ROC_AUC=roc_auc_score(y, prob),
                F1=f1_score(y, pred), Precision=precision_score(y, pred, zero_division=0),
                Recall=recall_score(y, pred), Threshold=thr)

# Persistence: use today's NO2 as the "score"; the decision rule is today > THRESHOLD
persist_val = X_val["no2_today"].values
persist_test = X_test["no2_today"].values
baseline = scores(y_test, persist_test, THRESHOLD)
pd.DataFrame([baseline], index=["Persistence"]).round(3)

## 10. Models
Same three model families as a classic tabular project, now wrapped in pipelines so that missing values and scaling are handled **inside** each training fold (no leakage):

1. **Logistic Regression** – linear benchmark (imputation + scaling + `class_weight="balanced"`).
2. **Random Forest** – bagging ensemble (imputation + `class_weight="balanced"`).
3. **XGBoost** – boosting ensemble; handles missing values natively; `scale_pos_weight` set from the class ratio.

Cross-validation uses **`TimeSeriesSplit`**: each fold trains on the past and validates on the period right after it.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, RandomizedSearchCV
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
tscv = TimeSeriesSplit(n_splits=5)

models = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
                                   RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                                          class_weight="balanced", n_jobs=-1,
                                                          random_state=RANDOM_STATE)),
    "XGBoost": XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=ratio, eval_metric="logloss",
                             random_state=RANDOM_STATE),
}

cv_rows = []
for name, m in models.items():
    s = cross_val_score(m, X_train, y_train, cv=tscv, scoring="average_precision")
    cv_rows.append({"Model": name, "CV PR-AUC mean": s.mean(), "CV PR-AUC std": s.std()})
pd.DataFrame(cv_rows).round(3)

### Hyperparameter search for XGBoost
A small randomized search, still with `TimeSeriesSplit`, optimising PR-AUC.

In [ ]:
search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=ratio, eval_metric="logloss", random_state=RANDOM_STATE),
    param_distributions={
        "n_estimators": [200, 400, 700],
        "learning_rate": [0.02, 0.05, 0.1],
        "max_depth": [2, 3, 4, 6],
        "min_child_weight": [1, 3, 5],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
    },
    n_iter=25, scoring="average_precision", cv=tscv, random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
print("Best CV PR-AUC:", round(search.best_score_, 3))
print("Best params:", search.best_params_)
models["XGBoost (tuned)"] = search.best_estimator_

## 11. Choosing the decision threshold
A classifier outputs a probability; turning it into a yes/no needs a threshold. The default 0.5 is rarely right with imbalanced data. We fit each model on the training years and pick, **on the validation year**, the threshold that maximises F1. The test year is not touched.

In [ ]:
def best_threshold(y, prob):
    p, r, t = precision_recall_curve(y, prob)
    f1 = 2 * p * r / np.clip(p + r, 1e-9, None)
    i = np.nanargmax(f1[:-1])
    return float(t[i])

fitted, thresholds = {}, {}
for name, m in models.items():
    if name != "XGBoost (tuned)":
        m.fit(X_train, y_train)
    fitted[name] = m
    thresholds[name] = best_threshold(y_val, m.predict_proba(X_val)[:, 1])
pd.Series(thresholds, name="threshold chosen on validation").round(3)

## 12. Results on the test year

In [ ]:
rows = {"Persistence (baseline)": baseline}
probs = {}
for name, m in fitted.items():
    probs[name] = m.predict_proba(X_test)[:, 1]
    rows[name] = scores(y_test, probs[name], thresholds[name])
results = pd.DataFrame(rows).T.sort_values("PR_AUC", ascending=False)
display(results.round(3))
best_name = results.drop(index="Persistence (baseline)").index[0]
print("Best model:", best_name)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, pr in probs.items():
    p, r, _ = precision_recall_curve(y_test, pr)
    axes[0].plot(r, p, lw=2, label=f"{name} (AP={average_precision_score(y_test, pr):.2f})")
    fpr, tpr, _ = roc_curve(y_test, pr)
    axes[1].plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_test, pr):.2f})")
bp = baseline
axes[0].scatter(bp["Recall"], bp["Precision"], color="black", zorder=5, label="Persistence")
axes[0].axhline(y_test.mean(), color="grey", ls=":", label="No-skill")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision–Recall (test year)")
axes[1].plot([0, 1], [0, 1], "k--"); axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (test year)")
for a in axes: a.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
pred_best = (probs[best_name] >= thresholds[best_name]).astype(int)
cm = confusion_matrix(y_test, pred_best)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Normal", "High"], yticklabels=["Normal", "High"])
plt.title(f"Confusion matrix – {best_name}"); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

## 13. What drives the prediction?
We use **permutation importance on the test year**: how much PR-AUC drops when a feature is shuffled. Unlike the built-in tree importances, it is measured on unseen data and works for any model.

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(fitted[best_name], X_test, y_test, scoring="average_precision",
                            n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
imp.head(12)[::-1].plot(kind="barh", figsize=(8, 5), color="#4c72b0")
plt.title(f"Permutation importance – {best_name}"); plt.xlabel("Drop in PR-AUC")
plt.show()

## 14. Export results for the portfolio
Saves the test-year probabilities, labels and metrics to a small JSON file, used by the interactive threshold demo on my portfolio.

In [ ]:
out = {
    "station": STATION, "threshold_ugm3": THRESHOLD, "test_year": int(TEST_YEAR),
    "best_model": best_name, "decision_threshold": round(thresholds[best_name], 4),
    "metrics": {k: {m: round(float(v), 4) for m, v in r.items()} for k, r in rows.items()},
    "top_features": [[k, round(float(v), 4)] for k, v in imp.head(8).items()],
    "test": {"date": [d.strftime("%Y-%m-%d") for d in X_test.index],
             "prob": [round(float(p), 4) for p in probs[best_name]],
             "label": [int(v) for v in y_test]},
}
with open("results_portfolio.json", "w") as f:
    json.dump(out, f)
print("Saved results_portfolio.json")
if IN_COLAB:
    files.download("results_portfolio.json")

## 15. Conclusions
*(Fill in after running with the real data.)*

1. **Best model and how much it beats persistence:** …
2. **Main drivers:** …
3. **Threshold trade-off:** at the chosen threshold, the model catches …% of high days (recall) and …% of its alerts are correct (precision).
4. **Practical use:** …

### Limitations and next steps
- **Weather forecast vs. observed weather:** the model uses tomorrow's *observed* weather as a stand-in for the forecast. In production, forecast errors would lower performance a little; the next step is to train with archived forecasts.
- **One station:** extend to all Madrid stations, either one model per station or one global model with the station as a feature.
- **Changing baseline:** NO₂ levels have dropped over the years (cleaner vehicles, low-emission zones), so the model should be retrained regularly.